# 21 Reading CSV

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Read CSV files reliably: set the right options (header, delimiter, quotes, encoding, nulls, dates), supply an explicit schema, read many files at once, and handle bad records with the <code>PERMISSIVE</code>, <code>DROPMALFORMED</code> and <code>FAILFAST</code> modes.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
CSV is still the most common format arriving from partners, exports and legacy systems, and the least reliable one: no types, inconsistent quoting, stray delimiters, different encodings, broken rows. How you read it decides whether bad data is caught at Bronze or silently corrupts Gold. Ingestion is about a fifth of the Data Engineer Associate exam.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 How Spark reads CSV</b><br>
<code>spark.read.format("csv").option(...).schema(...).load(path)</code>, or the shortcut <code>spark.read.csv(path, ...)</code>.
<ul><li><b>Path</b> can be one file, a folder, a glob (<code>/sales/2024-*.csv</code>) or a list of paths</li><li><b>header</b>: use the first line as column names (default <code>false</code>)</li><li><b>schema</b> vs <b>inferSchema</b>: an explicit schema is fast and safe. Inference reads the data an extra time and guesses</li><li><b>mode</b>: what to do with malformed rows: <code>PERMISSIVE</code> (default, keep with nulls), <code>DROPMALFORMED</code>, <code>FAILFAST</code></li><li><b>columnNameOfCorruptRecord</b>: keep the raw text of bad rows for debugging</li><li>Other common options: <code>sep</code>, <code>quote</code>, <code>escape</code>, <code>multiLine</code>, <code>nullValue</code>, <code>dateFormat</code>, <code>timestampFormat</code>, <code>encoding</code>, <code>ignoreLeadingWhiteSpace</code></li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A hospital network receives a daily <code>admissions_YYYYMMDD.csv</code> from 40 clinics. Some clinics use <code>;</code> as a separator, a few rows have notes containing commas and line breaks, and occasionally a row has a text value in the <code>age</code> column. The Bronze job reads all of the day's files with an explicit schema, <code>PERMISSIVE</code> mode and a corrupt-record column, then routes bad rows to a quarantine table instead of failing the whole load.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Free Edition has no DBFS root for your files. This notebook creates a <b>Unity Catalog volume</b> (<code>workspace.spark_basics.raw_files</code>) and writes the sample CSV files there with plain Python. Volumes are the recommended place for raw files on Databricks.
</div>

## Syntax

```python
df = (
    spark.read.format("csv")
    .option("header", True)
    .option("sep", ",")
    .option("mode", "PERMISSIVE")
    .schema("id INT, name STRING, amount DECIMAL(10,2), order_date DATE")
    .load("/Volumes/workspace/spark_basics/raw_files/orders/")
)

spark.read.csv(path, header=True, schema=schema)                  # shortcut
spark.sql("SELECT * FROM read_files('/Volumes/.../orders/', format => 'csv', header => true)")  # SQL (Databricks)
```

## Setup: create a volume and sample files

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a schema and a volume (if they don't exist yet), then writes several small CSV files: a clean one, one with a different delimiter, one with quoted fields and line breaks, and one with bad rows.<br><br>
<b>Why it matters</b><br>Every cell below reads real files, exactly as you would in a pipeline.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A printed <code>BASE</code> path such as <code>/Volumes/workspace/spark_basics/raw_files/lesson21</code>, then the list of files that were created.
</div>

In [0]:
CATALOG, SCHEMA, VOLUME = "workspace", "spark_basics", "raw_files"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/lesson21"
print("BASE =", BASE)

In [0]:
import os

files = {
    "orders/orders_2024_01.csv": """order_id,customer,amount,order_date,country
1,Amal,120.50,2024-01-05,IN
2,John,80.00,2024-01-06,UK
3,Vivek,300.00,2024-01-07,AE
""",
    "orders/orders_2024_02.csv": """order_id,customer,amount,order_date,country
4,Priya,45.25,2024-02-01,IN
5,Sara,NA,2024-02-03,AE
6,Ravi,99.99,2024-02-04,
""",
    "semicolon/stores.csv": """store_id;store_name;city
S1;Anna Nagar;Chennai
S2;Deira;Dubai
""",
    "quoted/reviews.csv": '''review_id,customer,comment
1,Amal,"Fast delivery, great price"
2,John,"Said ""excellent"" twice"
3,Vivek,"Line one
line two of the same comment"
''',
    "dirty/orders_bad.csv": """order_id,customer,amount,order_date,country
7,Meena,50.00,2024-03-01,IN
eight,Arun,60.00,2024-03-02,IN
9,Kiran,abc,2024-03-03,IN
10,Leela,70.00,03/04/2024,IN
11,Too,Many,Columns,Here,Extra
""",
}

for rel_path, content in files.items():
    full = f"{BASE}/{rel_path}"
    os.makedirs(os.path.dirname(full), exist_ok=True)
    with open(full, "w", encoding="utf-8") as f:
        f.write(content)

for root, _, names in os.walk(BASE):
    for n in sorted(names):
        print(os.path.join(root, n).replace(BASE, "BASE"))

## 1. The naive read: no options

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads one file with no options at all.<br><br>
<b>Why it matters</b><br>Without <code>header</code>, the header line becomes a data row and the columns are called <code>_c0</code>, <code>_c1</code>, ... And without a schema, every column is a <b>string</b>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
4 rows (including the header as data), columns <code>_c0</code> to <code>_c4</code>, all of type <code>string</code>.
</div>

In [0]:
naive = spark.read.csv(f"{BASE}/orders/orders_2024_01.csv")
naive.show()
naive.printSchema()

## 2. Header and schema inference

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Adds <code>header=True</code>, then <code>inferSchema=True</code>.<br><br>
<b>Why it matters</b><br>Inference makes types usable, but it costs an extra pass over the data and can guess wrong. Here, the <code>NA</code> in February's <code>amount</code> turns the whole column into a string.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
With inference on January only, <code>amount</code> is <code>double</code> and <code>order_date</code> is <code>date</code>. Reading both months, <code>amount</code> becomes <code>string</code> because of the <code>NA</code> value.
</div>

In [0]:
jan = spark.read.csv(f"{BASE}/orders/orders_2024_01.csv", header=True, inferSchema=True)
jan.printSchema()

both = spark.read.csv(f"{BASE}/orders/", header=True, inferSchema=True)
both.printSchema()
both.show()

## 3. Explicit schema and null handling (the production way)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads the whole <code>orders</code> folder with a DDL schema and tells Spark that <code>NA</code> means null.<br><br>
<b>Why it matters</b><br>An explicit schema skips inference, gives you the exact types (<code>DECIMAL</code> for money), and documents the contract. <code>nullValue</code> converts placeholder text into real nulls. An empty field is already read as null.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
6 rows with <code>amount</code> as <code>decimal(10,2)</code> and <code>order_date</code> as <code>date</code>. Sara's amount and Ravi's country are null.
</div>

In [0]:
orders_schema = "order_id INT, customer STRING, amount DECIMAL(10,2), order_date DATE, country STRING"

orders = (
    spark.read.format("csv")
    .option("header", True)
    .option("nullValue", "NA")
    .schema(orders_schema)
    .load(f"{BASE}/orders/")
)
orders.printSchema()
orders.orderBy("order_id").show()

## 4. Which file did each row come from?

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Adds the source file name and modification time from the hidden <code>_metadata</code> column.<br><br>
<b>Why it matters</b><br>Lineage at row level. When a value looks wrong, you can find the exact source file. In Bronze tables, always keep the file path and the load time.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Each order with <code>source_file</code> set to <code>orders_2024_01.csv</code> or <code>orders_2024_02.csv</code>, and a <code>file_modified</code> timestamp.
</div>

In [0]:
from pyspark.sql import functions as F

(
    orders.select(
        "*",
        F.col("_metadata.file_name").alias("source_file"),
        F.col("_metadata.file_modification_time").alias("file_modified"),
        F.current_timestamp().alias("ingested_at"),
    )
    .orderBy("order_id")
    .show(truncate=False)
)

## 5. Other delimiters, quotes and multi-line fields

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads a semicolon-separated file, then a file whose quoted fields contain commas, escaped quotes and a line break.<br><br>
<b>Why it matters</b><br>Commas inside quotes are handled by default. Line breaks inside quotes need <code>multiLine=True</code>, otherwise one record is split into two broken rows. Doubled quotes (<code>""</code>) inside a field need <code>escape='"'</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The stores file with 3 proper columns. The reviews file: without <code>multiLine</code>, review 3 is broken across two rows. With <code>multiLine</code> and <code>escape</code>, 3 clean rows, including <code>Said "excellent" twice</code>.
</div>

In [0]:
spark.read.csv(f"{BASE}/semicolon/stores.csv", header=True, sep=";").show()

print("Without multiLine:")
spark.read.csv(f"{BASE}/quoted/reviews.csv", header=True).show(truncate=False)

print("With multiLine and escape:")
(
    spark.read.option("header", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(f"{BASE}/quoted/reviews.csv")
    .show(truncate=False)
)

## 6. Bad records: PERMISSIVE, DROPMALFORMED, FAILFAST

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads the dirty file in all three modes. In permissive mode, a <code>_corrupt_record</code> column keeps the raw text of every row that didn't fit the schema.<br><br>
<b>Why it matters</b><br>This is the core of robust ingestion. <code>PERMISSIVE</code> plus a corrupt-record column lets you load good rows <b>and</b> keep evidence of bad rows. <code>DROPMALFORMED</code> loses data silently. <code>FAILFAST</code> stops the job at the first problem, which suits strict contracts.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Watch out</b><br>
The corrupt-record column must be declared <b>in the schema</b> as a <code>STRING</code>. Also, Spark only parses the columns a query needs (column pruning), so <b>counts can lie</b>: <code>count()</code> on a <code>DROPMALFORMED</code> read doesn't parse the values and still counts the bad rows.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
PERMISSIVE: 5 rows. Rows <code>eight</code>, <code>abc</code>, <code>03/04/2024</code> and the extra-columns row have nulls in the bad fields and their raw text in <code>_corrupt_record</code>. DROPMALFORMED: <code>count()</code> prints <b>5</b>, but <code>show()</code> displays only the <b>1</b> clean row (Meena). That is the column-pruning trap. FAILFAST: an error message.
</div>

In [0]:
schema_with_corrupt = orders_schema + ", _corrupt_record STRING"

permissive = (
    spark.read.option("header", True)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(schema_with_corrupt)
    .csv(f"{BASE}/dirty/")
)
permissive.show(truncate=False)

In [0]:
dropped = spark.read.option("header", True).option("mode", "DROPMALFORMED").schema(orders_schema).csv(f"{BASE}/dirty/")

# count() needs no columns, so Spark doesn't parse the values and can't tell which rows are malformed.
print("DROPMALFORMED count():", dropped.count())
# show() parses every column, so the malformed rows are actually dropped.
dropped.show()

try:
    spark.read.option("header", True).option("mode", "FAILFAST").schema(orders_schema).csv(f"{BASE}/dirty/").show()
except Exception as e:
    print("FAILFAST:", type(e).__name__, "-", str(e).splitlines()[0][:160])

## 7. Split good and bad rows (quarantine pattern)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lands the permissive read in a Bronze table (corrupt-record column included), then splits clean rows from bad ones by reading that table.<br><br>
<b>Why it matters</b><br>This is how production Bronze/Silver jobs keep running when a source sends a few broken rows, while still making the problem visible and measurable.<br><br>
There is also a technical reason to land the data first. Spark <b>refuses</b> queries on a raw CSV/JSON read that reference only the corrupt-record column (error <code>UNSUPPORTED_FEATURE.QUERY_ONLY_CORRUPT_RECORD_COLUMN</code>), because pruning would give wrong answers. The usual workaround, <code>cache()</code>, isn't available on serverless. Writing the parsed rows to a table first is the clean, production-grade fix.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
1 good row and 4 quarantined rows, a bad-row percentage of 80%, then the clean row and the quarantined raw lines.
</div>

In [0]:
# Land the parsed rows (including _corrupt_record) as a Bronze Delta table.
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.lesson21_orders_bronze"
permissive.write.mode("overwrite").saveAsTable(BRONZE_TABLE)
bronze = spark.table(BRONZE_TABLE)

In [0]:
classified = bronze.withColumn("is_bad", F.col("_corrupt_record").isNotNull())

good = classified.filter(~F.col("is_bad")).drop("_corrupt_record", "is_bad")
quarantine = classified.filter("is_bad").select("_corrupt_record", F.current_timestamp().alias("quarantined_at"))

total = classified.count()
bad = quarantine.count()
print(f"good: {total - bad} | quarantined: {bad} | bad %: {bad / total:.0%}")
good.show()
quarantine.show(truncate=False)

## 8. Dates in non-standard formats

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads dates written as <code>dd/MM/yyyy</code> with the <code>dateFormat</code> option.<br><br>
<b>Why it matters</b><br>Without the right <code>dateFormat</code>, the date can't be parsed and the row is treated as malformed (or the date becomes null).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A date of <code>2024-04-03</code> for the row written as <code>03/04/2024</code>.
</div>

In [0]:
import os
eu_path = f"{BASE}/eu_dates/sales.csv"
os.makedirs(os.path.dirname(eu_path), exist_ok=True)
with open(eu_path, "w") as f:
    f.write("sale_id,sale_date\n1,03/04/2024\n2,25/12/2024\n")

spark.read.csv(eu_path, header=True, schema="sale_id INT, sale_date DATE", dateFormat="dd/MM/yyyy").show()

## 9. Reading CSV with SQL on Databricks: `read_files`

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Queries the CSV folder directly from SQL with the <code>read_files</code> table-valued function.<br><br>
<b>Why it matters</b><br><code>read_files</code> is how Databricks SQL users (and many exam questions) read raw files without writing PySpark. It supports the same options and adds a <code>_rescued_data</code> column for values that don't fit the schema.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
<code>read_files</code> is a Databricks feature. It works in Free Edition, but not in open-source Spark.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The 6 orders, read with the schema hint.
</div>

In [0]:
display(spark.sql(f"""
    SELECT *
    FROM read_files(
        '{BASE}/orders/',
        format => 'csv',
        header => true,
        schema => 'order_id INT, customer STRING, amount DECIMAL(10,2), order_date DATE, country STRING'
    )
"""))

## Under the hood

```
spark.read.csv(folder, schema=...)
   1. List files in the folder (driver)
   2. Split files into chunks (~128 MB by default, spark.sql.files.maxPartitionBytes)
      -> one task per chunk, so big files are read in parallel
   3. Each task parses its chunk line by line into rows

with inferSchema=True:
   0. EXTRA JOB: read (a sample of) the data to guess each column's type

with multiLine=True:
   a file can't be split (a record may cross a split boundary) -> one task per FILE
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> with `inferSchema=True` or `header` detection on some runtimes, `spark.read` may start a job immediately, before you call any action.

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> with an explicit schema, the read is lazy: nothing happens until an action.

**Compressed files:** `.csv.gz` is read automatically, but gzip isn't splittable, so each file is one task. Many medium-sized files are better than one huge gzip.

Look at the plan below: a `FileScan csv` node with the schema and pushed filters.

In [0]:
orders.filter(F.col("country") == "IN").explain()

In [0]:
# Optional cleanup: remove this lesson's table and files.
spark.sql(f"DROP TABLE IF EXISTS {BRONZE_TABLE}")
dbutils.fs.rm(BASE, True)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: columns called <code>_c0</code>, <code>_c1</code> and the header appears as data</b><br>
Add <code>header=True</code>.<br><br>
<b>⛔ Problem: every column is a string</b><br>No schema and no <code>inferSchema</code>. Provide an explicit schema.<br><br>
<b>⛔ Problem: a numeric column is inferred as string</b><br>A placeholder such as <code>NA</code>, <code>-</code> or <code>null</code> is in the data. Use <code>nullValue</code> and an explicit schema.<br><br>
<b>⛔ Problem: rows are split or columns shifted</b><br>Quoted fields contain line breaks or delimiters. Use <code>multiLine=True</code>, and check <code>quote</code> and <code>escape</code>.<br><br>
**⛔ Problem: <code>_corrupt_record</code> is always null**<br>It must be in the schema as <code>STRING</code>, and the mode must be <code>PERMISSIVE</code>.<br><br>
**⛔ Problem: <code>QUERY_ONLY_CORRUPT_RECORD_COLUMN</code> error**<br>A query on a raw file read references only the corrupt-record column. Land the parsed rows in a table first and query that.<br><br>
**⛔ Problem: <code>count()</code> doesn't match what <code>show()</code> displays**<br>Column pruning: a count doesn't parse values, so malformed rows aren't detected. Validate counts after landing the data.<br><br>
**⛔ Problem: weird characters like <code>Ã©</code>**<br>Wrong encoding. Set <code>encoding</code> (for example <code>ISO-8859-1</code> or <code>UTF-16</code>).<br><br>
**⛔ Problem: <code>PATH_NOT_FOUND</code> or permission errors in Free Edition**<br>Use a <code>/Volumes/...</code> path. DBFS root paths such as <code>/FileStore</code> and <code>dbfs:/tmp</code> are not available.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. How do you read a CSV file in Spark, and which options matter most?</b><br>
<code>spark.read.format("csv")</code> with <code>header</code>, an explicit <code>schema</code>, <code>sep</code>, <code>mode</code>, <code>nullValue</code>, <code>dateFormat</code>, and <code>multiLine</code>/<code>quote</code>/<code>escape</code> for messy text. In production I always provide a schema.<br><br>

<b>🎤 2. Why avoid <code>inferSchema</code> in production?</b><br>
It needs an extra pass over the data, so it's slow on large files. It can infer the wrong type (for example a string because of one <code>NA</code>), and the schema can change between runs when the data changes, which breaks downstream code.<br><br>

<b>🎤 3. What are the CSV read modes?</b><br>
<code>PERMISSIVE</code> (default) keeps malformed rows with nulls and can store the raw line in a corrupt-record column. <code>DROPMALFORMED</code> silently drops them. <code>FAILFAST</code> raises an error on the first bad row.<br><br>

<b>🎤 4. How do you handle bad records without failing the pipeline?</b><br>
Read in <code>PERMISSIVE</code> mode with <code>columnNameOfCorruptRecord</code> in the schema, land the result in a Bronze table, then split good and bad rows: good rows continue, bad rows go to a quarantine table with a timestamp, and I monitor the bad-row rate. On Databricks, <code>badRecordsPath</code> or the rescued data column are alternatives.<br><br>

<b>🎤 5. How do you know which file a row came from?</b><br>
The hidden <code>_metadata</code> column, for example <code>_metadata.file_path</code> or <code>file_name</code>, plus <code>file_modification_time</code>. I store it in Bronze for lineage.<br><br>

<b>🎤 6. How does Spark parallelize reading CSV?</b><br>
It splits files into chunks of about 128 MB, with one task per chunk. Files with <code>multiLine=True</code> or gzip compression can't be split, so each one is read by a single task.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A CSV source occasionally contains malformed rows. The team must load all valid rows and keep the malformed rows for review without failing the job. Which read mode should they use?</b><br>
A. <code>FAILFAST</code><br>
B. <code>DROPMALFORMED</code><br>
C. <code>PERMISSIVE</code> with a corrupt record column<br>
D. <code>IGNORE</code><br>
<details><summary><b>Show answer</b></summary><b>C.</b> PERMISSIVE keeps the job running and the corrupt-record column preserves the raw text. DROPMALFORMED loses the bad rows, and IGNORE is not a read mode.</details><br><br>

<b>Q2. Which Databricks SQL function reads CSV files from a volume path directly in a query?</b><br>
A. <code>read_csv()</code><br>
B. <code>read_files()</code><br>
C. <code>LOAD DATA</code><br>
D. <code>spark_read()</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>read_files</code> is the table-valued function for reading files in SQL on Databricks.</details><br><br>

<b>Q3. Where should raw files be stored in a Unity Catalog-enabled workspace such as Free Edition?</b><br>
A. The DBFS root (<code>dbfs:/FileStore</code>)<br>
B. The driver's local disk<br>
C. A Unity Catalog volume (<code>/Volumes/catalog/schema/volume/...</code>)<br>
D. The notebook's folder<br>
<details><summary><b>Show answer</b></summary><b>C.</b> Volumes are governed storage for non-tabular files.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Write a new file <code>BASE/practice/products.csv</code> with a pipe (<code>|</code>) delimiter, a header, a <code>price</code> column where one value is <code>n/a</code>, and a <code>launch_date</code> in <code>dd-MM-yyyy</code> format</li><li>Read it with an explicit schema (<code>DECIMAL</code> price, <code>DATE</code> launch date) so that <code>n/a</code> becomes null and dates are parsed</li><li>Add the source file name and an ingestion timestamp</li><li>Using the <code>bronze</code> data from section 7, produce two DataFrames, <code>valid</code> and <code>rejected</code>, and print the counts</li><li>In a comment, explain why <code>inferSchema</code> would make <code>price</code> a string here</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
import os

# 1. Write the practice file
practice_path = f"{BASE}/practice/products.csv"
os.makedirs(os.path.dirname(practice_path), exist_ok=True)
with open(practice_path, "w") as f:
    f.write("product_id|name|price|launch_date\n1|Laptop|899.99|15-03-2024\n2|Mouse|n/a|01-04-2024\n")

# 2-3. Explicit schema, pipe delimiter, null placeholder, date format, lineage columns
products = (
    spark.read.option("header", True)
    .option("sep", "|")
    .option("nullValue", "n/a")
    .option("dateFormat", "dd-MM-yyyy")
    .schema("product_id INT, name STRING, price DECIMAL(10,2), launch_date DATE")
    .csv(practice_path)
    .select("*", F.col("_metadata.file_name").alias("source_file"), F.current_timestamp().alias("ingested_at"))
)
products.show(truncate=False)

# 4. Valid vs rejected, from the landed Bronze data (not the raw file read)
valid = bronze.filter(F.col("_corrupt_record").isNull()).drop("_corrupt_record")
rejected = bronze.filter(F.col("_corrupt_record").isNotNull())
print("valid:", valid.count(), "| rejected:", rejected.count())

# 5. With inferSchema, 'n/a' can't be parsed as a number, so Spark falls back to string for the whole column.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Always set <code>header</code> and an explicit <code>schema</code> in production. Inference is slow and fragile</li><li>Paths can be a file, a folder, a glob or a list. On Free Edition, use <code>/Volumes/...</code></li><li><code>nullValue</code>, <code>dateFormat</code>, <code>sep</code>, <code>quote</code>, <code>escape</code>, <code>multiLine</code> fix most messy files</li><li>Modes: <code>PERMISSIVE</code> (default) / <code>DROPMALFORMED</code> / <code>FAILFAST</code></li><li>Keep bad rows with <code>columnNameOfCorruptRecord</code>, land them in a table, then quarantine</li><li>Counts on raw reads can lie because of column pruning</li><li><code>_metadata.file_name</code> / <code>file_path</code> gives row-level lineage</li><li>Big files are split into ~128 MB tasks. <code>multiLine</code> and gzip files are not splittable</li><li>SQL on Databricks: <code>read_files(path, format =&gt; 'csv', ...)</code></li></ul>
</div>

| Task | Code |
|---|---|
| Read with schema | `spark.read.csv(path, header=True, schema=ddl)` |
| Delimiter | `.option("sep", ";")` |
| Null text | `.option("nullValue", "NA")` |
| Dates | `.option("dateFormat", "dd/MM/yyyy")` |
| Multi-line | `.option("multiLine", True).option("escape", '"')` |
| Bad rows | `.option("mode", "PERMISSIVE").option("columnNameOfCorruptRecord", "_corrupt_record")` |
| Lineage | `F.col("_metadata.file_name")` |

## Git commit

```
git add 01_spark_basics/21_reading_csv.ipynb
git commit -m "add 21 reading csv notebook"
```